## new

In [1]:
import pandas as pd

# Load datasets
us = pd.read_csv("/Users/ghadena/Desktop/geopol/data/raw/us_df.csv")
german = pd.read_parquet("/Users/ghadena/Desktop/geopol/data/raw/urls.parquet")
french = pd.read_parquet("/Users/ghadena/Desktop/geopol/data/raw/french_election_texts.parquet")
entities = pd.read_csv("/Users/ghadena/Desktop/geopol/data/processed/entites_n_relationships.csv")

# Add source tags
us['source'] = 'us'
german['source'] = 'german'
french['source'] = 'french'

# --- Step 1: US relevance flag ---
us['is_relevant'] = (us['relevance_text'] == 1).astype("Int64") 
# Rename text column
if 'text' in us.columns:
    us.rename(columns={'text': 'article_text'}, inplace=True)
us.drop(columns=[
    'is_us_election', 'is_french_election', 'is_german_election',
    'relevance_text', 'relevance_description', 'text', 'Unnamed: 0'
], inplace=True, errors='ignore')

# --- Step 2: Create relevance lookup from entities ---
relevance_lookup = entities[['url', 'relevant_to_german_or_french_elections']].copy()

# Clean and convert to boolean: "true" → 1, everything else → 0
# Step 2: Clean and convert to 0 / 1 / NaN
relevance_lookup['is_relevant'] = (
    relevance_lookup['relevant_to_german_or_french_elections']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: 1 if x == 'true' else (0 if x == 'false' else pd.NA))
).astype("Int64") 

# Keep only what we need
relevance_lookup = relevance_lookup[['url', 'is_relevant']]

# --- Step 3: Merge into German & French ---
german = german.merge(relevance_lookup, on='url', how='left')
french = french.merge(relevance_lookup, on='url', how='left')

# Do NOT fill nulls — let is_relevant be NaN if URL not in entities

# --- Step 4: Combine all articles ---
texts = pd.concat([us, french, german], ignore_index=True)


In [2]:
texts.is_relevant.value_counts(normalize=True)

is_relevant
0    0.821051
1    0.178949
Name: proportion, dtype: Float64

In [37]:
# --- 10. Check % relevant articles by source ---
relevance_by_source = (
    texts.groupby('source')['is_relevant']
    .mean()
    .mul(100)
    .round(2)
    .rename('% Relevant')
)
print("📊 % Relevant articles by source:")
print(relevance_by_source)

# --- 11. Check missing article_text (NaN or empty) ---
missing_text = texts[
    texts['article_text'].isna() |
    (texts['article_text'].astype(str).str.strip() == '')
]
print(f"\n🛑 Articles with empty text: {len(missing_text)}")


📊 % Relevant articles by source:
source
french    15.92
german     9.86
us        26.15
Name: % Relevant, dtype: Float64

🛑 Articles with empty text: 1404


In [3]:
texts.shape

(6587, 7)

In [43]:
print("US columns:", us.columns)
print("German columns:", german.columns)
print("French columns:", french.columns)

US columns: Index(['url', 'datetime', 'language', 'description', 'article_text', 'source',
       'is_relevant'],
      dtype='object')
German columns: Index(['url', 'datetime', 'language', 'description', 'article_text', 'source',
       'is_relevant'],
      dtype='object')
French columns: Index(['url', 'datetime', 'language', 'description', 'article_text', 'source',
       'is_relevant'],
      dtype='object')


In [4]:
null_articles = texts[
    texts['article_text'].isna() |
    (texts['article_text'].astype(str).str.strip() == '')
]
null_articles

,url,datetime,language,description,article_text,source,is_relevant
3,https://www.nytimes.com/video,2025-04-21,en,"Last December, video emerged showing the bodie...",NaN,us,0
32,https://www.cnn.com/2024/09/06/europe/video-ru...,2024-09-06,en,"The troops stagger onto a dusty track, then on...",NaN,us,0
123,https://www.reuters.com/world/europe/books-scr...,2024-09-10,en,"This autumn, pupils in the Finnish town of Rii...",NaN,us,0
264,https://www.npr.org/2024/09/19/g-s1-23689/insi...,2024-09-19,en,"As they fought to keep Sean ""Diddy"" Combs out ...",NaN,us,0
274,https://www.bbc.com/news/articles/cz04m913m49o,2024-09-20,en,Details about the walkie-talkies detonated in ...,NaN,us,0
...,...,...,...,...,...,...,...
6582,https://www.france24.com/en/live-news/20250420...,2025-04-20T00:00:00.0000000,en,Holger Rune beat world number two Carlos Alcar...,,german,0
6583,https://www.france24.com/en/live-news/20250411...,2025-04-11T00:00:00.0000000,en,Chinese President Xi Jinping urged the Europea...,,german,0
6584,https://www.france24.com/en/tv-shows/business/...,2025-04-16T00:00:00.0000000,en,Emboldened by the latest economic data that sh...,,german,0
6585,https://www.france24.com/en/tv-shows/down-to-e...,2025-04-07T00:00:00.0000000,en,Wildlife trafficking is one of the most profit...,,german,0


In [5]:
null_counts_by_source = null_articles['source'].value_counts()
print(null_counts_by_source)

source
french    1008
german     364
us          32
Name: count, dtype: int64


In [60]:
# Check if this URL had article_text in original german
sample_url = null_articles.iloc[4]['url']
print("us original:")
print(us[us['url'] == sample_url][['url', 'article_text']])

us original:
                                                url article_text
274  https://www.bbc.com/news/articles/cz04m913m49o          NaN


In [61]:
null_articles.is_relevant.value_counts()

is_relevant
0    995
1    102
Name: count, dtype: Int64

In [63]:
# Flag articles where article_text is missing or empty
texts['used_description_as_text'] = (
    texts['article_text'].isna() |
    (texts['article_text'].astype(str).str.strip() == '')
).astype(int)

In [64]:
# Check how many and what sources have missing relevance info
is_relevant_nulls = texts[texts['is_relevant'].isna()]
print(f"🔍 Articles with null is_relevant: {len(is_relevant_nulls)}")

# Count by source
print("By source:")
print(is_relevant_nulls['source'].value_counts())

# Preview
is_relevant_nulls[['url', 'source', 'language', 'article_text', 'description']].head()

🔍 Articles with null is_relevant: 535
By source:
source
french    443
german     92
Name: count, dtype: int64


,url,source,language,article_text,description
3763,https://www.reuters.com/business/tariffs/,french,en,,Reuters.com is your online source for the late...
3764,https://www.reuters.com/world/russia-united-st...,french,en,,The United States and Russia both said on Thur...
3765,https://www.reuters.com/business/media-telecom...,french,en,,Item 1 of 6 Director Richard Linklater attends...
3766,https://www.reuters.com/world/europe/russia-us...,french,en,,"WASHINGTON/MOSCOW/KYIV, March 25 (Reuters) - T..."
3767,https://www.reuters.com/markets/europe/ukraine...,french,en,,Ukrainian service personnel use searchlights a...


In [70]:
is_relevant_nulls.sample(5).description.to_list()

['His comments come as Merkel, who led Germany between 2005 and 2021, re-enters the political fray. In a rare intervention, Merkel criticized Christian Democratic Union (CDU) leader Friedrich Merz for breaking the party’s long-standing firewall against the AfD by allowing far-right votes to help pass an anti-immigration motion in the Bundestag.. The nonbinding motion called for the rejection ...',
 'The Associated Press is an independent global news organization dedicated to factual reporting. Founded in 1846, AP today remains the most trusted source of fast, accurate, unbiased news in all formats and the essential provider of the technology and services vital to the news business.',
 'Shortly before the second round, Romania’s Constitutional Court ruled that the first round of voting had been so badly tainted by a Russian influence operation on social media that the entire process had to be scrapped and started over.. Iohannis was allowed to remain in office until the rerun of the pre

In [ ]:
is_relevant_nulls.to_csv("redo2.csv") ## to do tmr 

# droped is relevant nulls - merge back later 

In [6]:
allowed_langs = ['en', 'de', 'fr']
texts = texts[texts['language'].isin(allowed_langs)].reset_index(drop=True)

In [7]:
texts = texts[texts['is_relevant'].notna()].reset_index(drop=True)

In [8]:
texts.shape

(6050, 7)

# merge w entites 

In [9]:
# Load entities if not already in memory
entities = pd.read_csv("/Users/ghadena/Desktop/geopol/data/processed/entites_n_relationships.csv")

# Drop any duplicate URLs in entities (just to be safe)
entities = entities.drop_duplicates(subset='url')

# Merge texts with entities on 'url'
merged = texts.merge(entities, on='url', how='left', indicator=True)

In [16]:
entities.relevant_to_german_or_french_elections.value_counts()
entities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6250 entries, 0 to 6249
Data columns (total 6 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   Unnamed: 0                              6250 non-null   int64 
 1   url                                     6250 non-null   object
 2   lang                                    3981 non-null   object
 3   extracted_entities                      6250 non-null   object
 4   entity_relationships                    6250 non-null   object
 5   relevant_to_german_or_french_elections  3527 non-null   object
dtypes: int64(1), object(5)
memory usage: 293.1+ KB


In [198]:
merged["datetime"] = merged["datetime"].apply(
    lambda x: pd.to_datetime(x, errors='coerce') if pd.notna(x) else pd.NaT
)

In [199]:
merged.datetime

0      2024-09-12
1      2024-09-09
2      2024-09-11
3      2025-04-21
4      2024-09-09
          ...    
6045   2025-04-20
6046   2025-04-11
6047   2025-04-16
6048   2025-04-07
6049   2025-04-16
Name: datetime, Length: 6050, dtype: datetime64[ns]

In [200]:
# Check how many articles didn't get matched
unmatched = merged[merged['_merge'] == 'left_only']
print(f"🟡 Unmatched articles: {len(unmatched)}")


🟡 Unmatched articles: 47


In [201]:
import pandas as pd
import json

# 1. Drop unmatched rows (only keep rows that were matched during merge)
merged = merged[merged['_merge'] == 'both'].reset_index(drop=True)

# 2. Drop unneeded columns
merged.drop(columns=[
    'Unnamed: 0', 'lang', 'relevant_to_german_or_french_elections', '_merge'
], inplace=True, errors='ignore')


In [202]:
# 3. Add flags: has_entities and has_relationships
import json
import ast

def is_non_empty_entity_dict(value):
    try:
        data = ast.literal_eval(value) if isinstance(value, str) else value
        if isinstance(data, dict):
            return any(len(v) > 0 for v in data.values())
        return False
    except (ValueError, SyntaxError):
        return False

def is_non_empty_relationship_list(value):
    try:
        data = ast.literal_eval(value) if isinstance(value, str) else value
        return isinstance(data, list) and len(data) > 0
    except (ValueError, SyntaxError):
        return False

# Apply to your merged DataFrame
merged['has_entities'] = merged['extracted_entities'].apply(is_non_empty_entity_dict).astype(int)
merged['has_relationships'] = merged['entity_relationships'].apply(is_non_empty_relationship_list).astype(int)

In [203]:
import ast

def has_key_in_literal_dict(field, key):
    try:
        obj = ast.literal_eval(field) if isinstance(field, str) else field
        return int(bool(obj.get(key)))
    except (ValueError, SyntaxError, TypeError):
        return 0
    
merged['has_people'] = merged['extracted_entities'].apply(lambda x: has_key_in_literal_dict(x, 'people'))
merged['has_organizations'] = merged['extracted_entities'].apply(lambda x: has_key_in_literal_dict(x, 'institutions'))

In [204]:
merged 

,url,datetime,language,description,article_text,source,is_relevant,used_description_as_text,extracted_entities,entity_relationships,has_entities,has_relationships,has_people,has_organizations
0,https://www.washingtonpost.com/politics/intera...,2024-09-10,en,A: Trump believes human activity is just one c...,Q: Do you believe that climate change is large...,us,1,0,"{'people': ['Trump'], 'institutions': ['The Wa...","[{'source': 'Trump', 'target': 'climate change...",1,1,1,1
1,https://www.washingtonpost.com/politics/2024/0...,2024-09-09,en,"The page lists Harris’s economic, immigration ...",Vice President Kamala Harris’s campaign posted...,us,1,0,"{'people': ['Kamala Harris', 'Donald Trump', '...","[{'source': 'Kamala Harris', 'target': 'Donald...",1,1,1,1
2,https://www.washingtonpost.com/politics/intera...,2024-09-10,en,A: Harris wants to strengthen Social Security ...,Q: Do you believe that climate change is large...,us,1,0,"{'people': ['Harris'], 'institutions': ['Unite...","[{'source': 'Harris', 'target': 'climate chang...",1,1,1,1
3,https://www.washingtonpost.com/politics/2024/0...,2024-09-11,en,"Republicans in Winnemucca, Nev., gather at a w...",So just how emphatic was her win? And what do ...,us,1,0,"{'people': ['Harris', 'Trump', 'Joe Biden', 'H...","[{'source': 'Joe Biden', 'target': 'Trump', 'r...",1,1,1,1
4,https://www.washingtonpost.com/politics/intera...,2024-09-10,en,"Kamala Harris’s immigration policies, explaine...",Deporting undocumented people Q: Should all u...,us,1,0,"{'people': ['Harris', 'Donald Trump', 'Biden']...","[{'source': 'Harris', 'target': 'Donald Trump'...",1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,https://www.france24.com/en/live-news/20250420...,2025-04-20,en,Holger Rune beat world number two Carlos Alcar...,,german,0,1,"{'people': ['Holger Rune', 'Carlos Alcaraz'], ...",[],1,0,1,0
5999,https://www.france24.com/en/live-news/20250411...,2025-04-11,en,Chinese President Xi Jinping urged the Europea...,,german,0,1,"{'people': ['Xi Jinping', 'Donald Trump'], 'in...","[{'source': 'Xi Jinping', 'target': 'European ...",1,1,1,1
6000,https://www.france24.com/en/tv-shows/business/...,2025-04-16,en,Emboldened by the latest economic data that sh...,,german,0,1,"{'people': [], 'institutions': [], 'political_...",[],1,0,0,0
6001,https://www.france24.com/en/tv-shows/down-to-e...,2025-04-07,en,Wildlife trafficking is one of the most profit...,,german,0,1,"{'people': [], 'institutions': [], 'political_...",[],0,0,0,0


In [205]:
both = merged[(merged['has_entities'] == 1) & (merged['has_relationships'] == 1)]
only_entities = merged[(merged['has_entities'] == 1) & (merged['has_relationships'] == 0)]
only_relationships = merged[(merged['has_entities'] == 0) & (merged['has_relationships'] == 1)]
neither = merged[(merged['has_entities'] == 0) & (merged['has_relationships'] == 0)]

print("🧾 Article extraction summary:")
print(f"🟢 Both entities & relationships: {len(both)}")
print(f"🟡 Only entities: {len(only_entities)}")
print(f"🔵 Only relationships: {len(only_relationships)}")
print(f"🔴 Neither: {len(neither)}")

🧾 Article extraction summary:
🟢 Both entities & relationships: 4180
🟡 Only entities: 1519
🔵 Only relationships: 0
🔴 Neither: 304


In [206]:
print("\n🧠 Breakdown of specific entity types:")
print("People:", merged['has_people'].sum())
print("Organizations:", merged['has_organizations'].sum())


🧠 Breakdown of specific entity types:
People: 4400
Organizations: 4912


# FE 

In [207]:
import nltk
nltk.download('punkt', download_dir='/Users/ghadena/nltk_data/tokenizers/punkt')
nltk.data.path.append('/Users/ghadena/nltk_data/tokenizers/punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ghadena/nltk_data/tokenizers/punkt...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ghadena/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/ghadena/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [208]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from datetime import datetime
from nltk import pos_tag

# Use regex tokenizer to avoid punkt issues
def tokenize(text):
    return re.findall(r'\b\w+\b', text)

def count_verbs(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    try:
        tokens = tokenize(text)
        tags = pos_tag(tokens)
        return sum(1 for _, tag in tags if tag.startswith("VB"))  # VB, VBD, VBG, etc.
    except Exception:
        return 0

# def get_article_type(text):
#     if not isinstance(text, str): return "other"
#     intro = text[:200].lower()
#     if "analysis" in intro:
#         return "analysis"
#     if "interview with" in intro or intro.startswith("interview:"):
#         return "interview"
#     return "other"

def generate_article_features(
    df,
    text_col="article_text",
    url_col="url",
    date_col="datetime"
):
    df = df.copy()

    # --- Ensure datetime format
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", utc = False)

    # --- News org name from URL
    df["news_org"] = df[url_col].apply(lambda x: urlparse(x).netloc if pd.notnull(x) else None)

    # --- Extract slug-style title from URL
    df["url_slug_title"] = df[url_col].apply(
    lambda url: (
        url.strip("/").split("/")[-2].replace("-", " ").title()
        if pd.notnull(url) and "cnn.com" in url and url.strip("/").endswith("index.html")
        else url.strip("/").split("/")[-1].replace("-", " ").title()
        if pd.notnull(url) else None)
    )

    # --- Date features
    df["date"] = df[date_col].dt.date
    df["day_of_week"] = df[date_col].dt.day_name()
    df["month"] = df[date_col].dt.month
    df["is_weekend"] = df[date_col].dt.weekday >= 5

    # --- Days until elections
    us_election_date = pd.to_datetime("2024-11-05")
    french_election_date = pd.to_datetime("2024-07-07")
    german_election_date = pd.to_datetime("2025-02-23")
    df["days_until_us_election"] = (us_election_date - df[date_col]).dt.days
    df["days_until_french_election"] = (french_election_date - df[date_col]).dt.days
    df["days_until_german_election"] = (german_election_date - df[date_col]).dt.days

    # --- Word count
    df["word_count"] = df[text_col].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

    # --- Sentence count (basic estimate using punctuation)
    df["sentence_count"] = df[text_col].apply(lambda x: len(re.findall(r'[.!?]', str(x))) if pd.notnull(x) else 0)

    # --- Avg sentence length
    df["avg_sentence_length"] = df.apply(
        lambda row: row["word_count"] / row["sentence_count"] if row["sentence_count"] > 0 else 0,
        axis=1
    )

    # --- Noun count
    df["noun_count"] = df[text_col].apply(
        lambda text: sum(1 for word, tag in pos_tag(tokenize(str(text))) if tag.startswith("NN")) if pd.notnull(text) else 0
    )

    # --- Verb count
    df["verb_count"] = df[text_col].apply(count_verbs)
    
    # --- 
    df["contains_question"] = df[text_col].str.contains(r"\?", na=False)
    df["quote_count"] = df[text_col].apply(lambda x: str(x).count('"') + str(x).count("“"))
    df["contains_number"] = df[text_col].str.contains(r"\d", na=False)
    df["capital_ratio"] = df[text_col].apply(
    lambda x: sum(w.isupper() for w in str(x).split()) / len(str(x).split()) if isinstance(x, str) and len(x.split()) > 0 else 0
)
    social_platforms = ["twitter", "facebook", "instagram", "tiktok", "youtube", "threads", "x.com"]

    df["mentions_social_media"] = df[text_col].apply(
    lambda x: any(p in str(x).lower() for p in social_platforms)
    )

    # # --- Language fallback
    # if "language" not in df.columns:
    #     df["language"] = "en"

    # # --- Article type classification (simple heuristic)
    # df["article_type"] = df[text_col].apply(get_article_type)

    return df

In [210]:
df_featurized = generate_article_features(merged)

In [236]:
# dropped 16 obs wo date - only 1 was relevant 
df_featurized = df_featurized[df_featurized["datetime"].notna()].reset_index(drop=True)

In [212]:
df_featurized.groupby("is_relevant")[["contains_question", "contains_number", "mentions_social_media"]].describe().T
#df_featurized.groupby("relevance_text")[["capital_ratio", "quote_count"]].describe().T

is_relevant                       0      1
contains_question     count    4931   1072
                      unique      2      2
                      top     False  False
                      freq     3860    671
contains_number       count    4931   1072
                      unique      2      2
                      top      True   True
                      freq     3109    856
mentions_social_media count    4931   1072
                      unique      2      2
                      top     False  False
                      freq     4651    981

In [213]:
# Count how many unique news sources there are
unique_publishers = df_featurized['news_org'].nunique()
print(f"Number of unique publishers: {unique_publishers}")
df_featurized.news_org.value_counts()

Number of unique publishers: 27


news_org
www.aljazeera.com         665
www.politico.eu           641
www.euronews.com          590
www.france24.com          483
www.bbc.com               422
www.reuters.com           383
www.nbcnews.com           254
www.npr.org               218
www.cnn.com               209
www.liberation.fr         170
www.tagesschau.de         170
www.sueddeutsche.de       170
www.faz.net               170
www.spiegel.de            170
www.lefigaro.fr           170
www.zeit.de               170
www.nytimes.com           170
www.bfmtv.com             169
www.rfi.fr                169
www.bild.de               169
www.lemonde.fr            168
www.washingtonpost.com     66
www.bloomberg.com          18
apnews.com                 12
observers.france24.com      4
france24.com                2
www.theguardian.com         1
Name: count, dtype: int64

In [214]:
df_featurized.groupby("news_org")["is_relevant"].mean().sort_values(ascending=False)


news_org
www.nbcnews.com           0.602362
www.washingtonpost.com    0.575758
www.npr.org               0.454128
www.bloomberg.com         0.388889
www.liberation.fr         0.358824
www.cnn.com               0.354067
www.lefigaro.fr           0.335294
www.nytimes.com           0.294118
www.bfmtv.com             0.254438
www.tagesschau.de         0.229412
www.sueddeutsche.de       0.217647
www.rfi.fr                0.183432
www.zeit.de               0.176471
www.bild.de               0.147929
www.lemonde.fr               0.125
www.politico.eu           0.107644
www.aljazeera.com         0.106767
www.euronews.com               0.1
www.spiegel.de                 0.1
www.france24.com          0.093168
www.bbc.com                0.07346
www.faz.net               0.047059
www.reuters.com           0.018277
france24.com                   0.0
www.theguardian.com            0.0
observers.france24.com         0.0
apnews.com                     0.0
Name: is_relevant, dtype: Float64

In [215]:
df_featurized.columns

Index(['url', 'datetime', 'language', 'description', 'article_text', 'source',
       'is_relevant', 'used_description_as_text', 'extracted_entities',
       'entity_relationships', 'has_entities', 'has_relationships',
       'has_people', 'has_organizations', 'news_org', 'url_slug_title', 'date',
       'day_of_week', 'month', 'is_weekend', 'days_until_us_election',
       'days_until_french_election', 'days_until_german_election',
       'word_count', 'sentence_count', 'avg_sentence_length', 'noun_count',
       'verb_count', 'contains_question', 'quote_count', 'contains_number',
       'capital_ratio', 'mentions_social_media', 'article_type'],
      dtype='object')

In [216]:
df_featurized.groupby("is_relevant")[["word_count", "sentence_count", "avg_sentence_length", "noun_count", "verb_count"]].describe().T

is_relevant                           0             1
word_count          count   4931.000000   1072.000000
                    mean     567.107686    730.545709
                    std     1432.375152   1004.196277
                    min        0.000000      0.000000
                    25%       24.000000    149.750000
                    50%      216.000000    474.500000
                    75%      528.000000   1018.000000
                    max    15378.000000  14489.000000
sentence_count      count   4931.000000   1072.000000
                    mean      30.346583     39.120336
                    std       81.524673     58.393490
                    min        0.000000      0.000000
                    25%        2.000000      7.000000
                    50%       11.000000     24.000000
                    75%       26.000000     52.000000
                    max     1019.000000    710.000000
avg_sentence_length count   4931.000000   1072.000000
                    mean      16.159655     18.605533
                    std       10.173588      9.866474
                    min        0.000000      0.000000
                    25%       11.120192     14.574178
                    50%       17.694444     19.184659
                    75%       22.818182     23.625866
                    max       84.000000    171.333333
noun_count          count   4931.000000   1072.000000
                    mean     307.530116    308.869403
                    std     1041.354574    489.934716
                    min        0.000000      0.000000
                    25%       16.000000     71.000000
                    50%       91.000000    220.500000
                    75%      230.500000    400.250000
                    max    11239.000000   9289.000000
verb_count          count   4931.000000   1072.000000
                    mean      73.057189    111.140858
                    std      136.017765    152.721333
                    min        0.000000      0.000000
                    25%        3.000000     14.000000
                    50%       28.000000     56.500000
                    75%       81.000000    163.250000
                    max     1447.000000   1601.000000

# election words 


In [217]:
import re

def find_trigger_in_middle_body(row, keywords, cutoff_ratio=0.1):
    # Choose fallback if article_text is null or fallback flag is 1
    text = row['article_text']
    if pd.isna(text) or row.get('used_description_as_text', 0) == 1:
        text = row.get('description')

    if not isinstance(text, str):
        return None
    
    words = text.split()
    if len(words) < 5:
        return None

    start = int(len(words) * cutoff_ratio)
    end = int(len(words) * (1 - cutoff_ratio))
    middle_text = " ".join(words[start:end])
    
    for kw in keywords:
        if re.search(rf"\b{re.escape(kw)}\b", middle_text, flags=re.IGNORECASE):
            return kw
    return None

In [218]:

election_keywords = [
    "us election", "2024 election", "vote", "voting", "presidential campaign"
]

# Count number of matched election keywords in the article_text (or fallback)
df_featurized["election_keyword_count"] = df_featurized["article_text"].apply(
    lambda text: sum(
        bool(re.search(rf"\b{re.escape(kw)}\b", str(text), flags=re.IGNORECASE))
        for kw in election_keywords
    )
)

# Find if any keyword is mentioned in the middle of the text (with fallback support)
df_featurized["triggered_keyword"] = df_featurized.apply(
    lambda row: find_trigger_in_middle_body_rowwise(row, election_keywords),
    axis=1
)

# Flag if any keyword was found
df_featurized["mentions_election_keywords"] = df_featurized["triggered_keyword"].notna()

In [219]:
false_positives = df_featurized[
    (df_featurized["mentions_election_keywords"]) &
    (df_featurized["is_relevant"] == 0)
]

print(f"Election keyword false positives: {len(false_positives)}")


true_positives = df_featurized[
    (df_featurized["mentions_election_keywords"]) &
    (df_featurized["is_relevant"] == 1)
]

print(f"Election keyword true positives: {len(true_positives)}")
#true_positives[["article_text", "mentions_election_keywords", "is_relevant"]].head()




Election keyword false positives: 209
Election keyword true positives: 342


In [220]:
false_positives[["article_text", "mentions_election_keywords", "is_relevant", "triggered_keyword"]].head()
false_positives.iloc[1]["article_text"]

'Mexico\'s Senate just approved changing the constitution. Here\'s what you need to know  toggle caption Felix Marquez/AP  MEXICO CITY — Mexico’s Senate on Wednesday narrowly passed sweeping changes to the courts that include having judges elected by the public rather than appointed, in a major and controversial set of constitutional reforms.  The approval came hours after hundreds of protesters broke into Mexico\'s Senate, forcing the body to take a temporary recess. The proposed reforms have led judges and other judicial staff to strike and protest, in what’s become one of Mexico’s biggest constitutional debates in years.  Here are the main things to understand about the reforms and why they are so controversial.  Sponsor Message  The government vows to root out court corruption  For nearly a year, outgoing President Andrés Manuel López Obrador has been promoting a plan to remake the federal judiciary and Claudia Sheinbaum, the president-elect, due to take over in October, backs the 

In [221]:
df_featurized.is_relevant.value_counts()

is_relevant
0    4931
1    1072
Name: count, dtype: Int64

# try 2 for keyword flag 

In [222]:
# --- ENGLISH ---
english_keywords = {
    "election", "elections", "vote", "voting", "ballot", 
    "campaign", "candidate", "debate", "run for president"
}
english_figures = {
    "biden", "kamala", "harris", "trump", "pence", "obama", 
    "de santis", "haley", "kennedy", "bernie", "democrat", "republican"
}

# --- GERMAN ---
german_keywords = {
    "wahl", "wahlen", "abstimmung", "stimme", "abgeordnetenhaus", 
    "kanzler", "kandidaten", "wahlkampf", "bundestagswahl"
}
german_figures = {
    "scholz", "merkel", "laschet", "habeck", "baerbock", 
    "afd", "spd", "fdp", "grünen", "cdu", "csu"
}

# --- FRENCH ---
french_keywords = {
    "élection", "élections", "vote", "voter", "scrutin", 
    "candidat", "campagne", "présidentielle", "second tour"
}
french_figures = {
    "macron", "le pen", "mélenchon", "zemmour", "hidalgo", 
    "rn", "lrem", "ps", "républicains"
}

all_keywords = english_keywords | german_keywords | french_keywords
all_figures = english_figures | german_figures | french_figures

In [223]:
def compute_election_certainty(row, keywords, figures):
    text = row.get("article_text")
    
    # Fallback to description if article_text is missing
    if pd.isna(text) or row.get("used_description_as_text", 0) == 1:
        text = row.get("description")
    
    if not isinstance(text, str):
        return 0

    text_lower = text.lower()

    found_keywords = {kw for kw in keywords if re.search(rf"\b{re.escape(kw)}\b", text_lower)}
    found_figures = {pf for pf in figures if re.search(rf"\b{re.escape(pf)}\b", text_lower)}

    # High certainty if 2+ election keywords OR strong phrases
    if len(found_keywords) >= 2:
        return 2
    
    # Medium certainty if 1 keyword + a political figure mentioned
    if len(found_keywords) >= 1 and len(found_figures) >= 1:
        return 1

    # Otherwise, not election-related
    return 0

In [224]:
df_featurized['election_certainty'] = df_featurized.apply(
    lambda row: compute_election_certainty(row, all_keywords, all_figures),
    axis=1
)

In [225]:
print("🔍 Certainty counts:")
print(df_featurized['election_certainty'].value_counts())

# Breakdown
print("\n✅ High certainty:", (df_featurized['election_certainty'] == 2).sum())
print("⚠️ Medium certainty:", (df_featurized['election_certainty'] == 1).sum())
print("❌ Not election-related:", (df_featurized['election_certainty'] == 0).sum())

🔍 Certainty counts:
election_certainty
0    4792
2     841
1     370
Name: count, dtype: int64

✅ High certainty: 841
⚠️ Medium certainty: 370
❌ Not election-related: 4792


In [226]:
# High Certainty (2)
high = df_featurized[df_featurized['election_certainty'] == 2]
high_tp = high[high['is_relevant'] == 1]
high_fp = high[high['is_relevant'] == 0]

# Medium Certainty (1)
medium = df_featurized[df_featurized['election_certainty'] == 1]
medium_tp = medium[medium['is_relevant'] == 1]
medium_fp = medium[medium['is_relevant'] == 0]

# False Negatives: was relevant, but flagged as 0
fn = df_featurized[(df_featurized['election_certainty'] == 0) & (df_featurized['is_relevant'] == 1)]

# True Negatives: both are zero
tn = df_featurized[(df_featurized['election_certainty'] == 0) & (df_featurized['is_relevant'] == 0)]

In [227]:
print("📊 High Certainty")
print("✅ True Positives:", len(high_tp))
print("❌ False Positives:", len(high_fp))

print("\n📊 Medium Certainty")
print("✅ True Positives:", len(medium_tp))
print("❌ False Positives:", len(medium_fp))

print("\n📉 Missed Detections (False Negatives):", len(fn))
print("✅ True Negatives:", len(tn))

📊 High Certainty
✅ True Positives: 542
❌ False Positives: 299

📊 Medium Certainty
✅ True Positives: 185
❌ False Positives: 185

📉 Missed Detections (False Negatives): 345
✅ True Negatives: 4447


In [228]:
def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0

print("\n🎯 High Certainty Precision:", round(precision(len(high_tp), len(high_fp)), 2))
print("🎯 Medium Certainty Precision:", round(precision(len(medium_tp), len(medium_fp)), 2))

print("\n📈 Overall Recall:", round(recall(len(high_tp) + len(medium_tp), len(fn)), 2))


🎯 High Certainty Precision: 0.64
🎯 Medium Certainty Precision: 0.5

📈 Overall Recall: 0.68


We implemented a rule-based system to flag election-related content with high and medium certainty using multilingual keywords and political figure references. The high-certainty flags achieved a precision of 64%, meaning over a third of those articles were false positives, while the overall recall of 68% indicates that the rules missed about a third of truly relevant articles. These results suggest that while the rules are useful for capturing obvious election-related cases, they struggle with nuanced or indirect mentions—making them valuable but imperfect features for our downstream machine learning model, which can learn to generalize beyond these limitations.

# title 

In [229]:
df_featurized.url_slug_title.to_list()

['Donald Trump Policy Positions',
 'Kamala Harris Campaign Platform Website',
 'Kamala Harris Policy Positions',
 'Kamala Harris Debate Performance Polls',
 'Kamala Harris Immigration',
 'Change Healthcare Letter Hack Data Breach',
 'Nexus Yuval Noah Harari Review',
 'Liane Moriarty Here One Moment',
 'Kamala Harris Abortion',
 'Elizabeth Strout Tell Me Everything Review',
 'Waymo Vs Uber Lyft Cost Speed Robotaxi Rideshare',
 'Playground Richard Powers Novel Review',
 'Ukraine Thermite Dragon Drones Intl Hnk Ml',
 'Debate Takeaways Trump Harris',
 'Nasa Boeing Starliner Capsule 09 06 24',
 'Trump Modify 25Th Amendment Harris Biden',
 'Trump Economic Plans Musk Government Commission',
 'Covid Vaccine Cost Pharmacies',
 'Boeing Starliner Return Without Astronauts',
 'Fact Check Debate Trump Harris',
 'Kamala Harris Donald Trump Debate',
 'Harris Trump Debate Analysis',
 'Fact Check Trump Vance Tariffs',
 'Rancho Palos Verdes Landslide Rainfall',
 'Rancho Palos Verdes California Landslide

In [230]:
# 1. Remove .html/.htm endings
df_featurized["url_slug_title"] = df_featurized["url_slug_title"].str.replace(r"\.html?$", "", case=False, regex=True)

# 2. Expand no_title to include:
#    - known opaque publishers (BBC, NPR)
#    - missing or empty titles
#    - titles with ≤ 1 word
df_featurized["no_title"] = (
    df_featurized["news_org"].isin(["www.bbc.com", "www.npr.org"]) |
    df_featurized["url_slug_title"].isna() |
    (df_featurized["url_slug_title"].str.strip() == "") |
    (df_featurized["url_slug_title"].str.strip().str.split().str.len() <= 1)
)


In [257]:
df_featurized["L_num_title_words"] = df_featurized["url_slug_title"].apply(
    lambda x: len(str(x).strip().split()) if pd.notna(x) else 0
)

In [259]:
df_featurized["L_num_title_words"].max()

np.int64(30)

In [232]:
df_featurized.date

0       2024-09-10
1       2024-09-09
2       2024-09-10
3       2024-09-11
4       2024-09-10
           ...    
5998    2025-04-20
5999    2025-04-11
6000    2025-04-16
6001    2025-04-07
6002    2025-04-16
Name: date, Length: 6003, dtype: object

In [233]:
df_featurized[df_featurized["date"].isna()]

,url,datetime,language,description,article_text,source,is_relevant,used_description_as_text,extracted_entities,entity_relationships,...,quote_count,contains_number,capital_ratio,mentions_social_media,article_type,election_keyword_count,triggered_keyword,mentions_election_keywords,election_certainty,no_title
3306,https://www.bbc.com/news/live/ce3qnyr7y94t,NaT,en,Donald Trump says he has no plans to pause glo...,Trump passes on pausing tariffs as global turm...,french,0,0,"{'people': ['Donald Trump', 'Natalie Sherman']...","[{'source': 'Donald Trump', 'target': 'China',...",...,6,True,0.027027,False,other,0,None,False,0,True
3491,https://www.aljazeera.com/where/kosovo,NaT,en,Stay on top of Kosovo latest developments on t...,An escalation into a conflict in the Western B...,french,0,0,"{'people': [], 'institutions': [], 'political_...",[],...,0,False,0.000000,False,other,0,None,False,0,True
3594,https://www.aljazeera.com/where/mexico,NaT,en,Stay on top of Mexico latest developments on t...,Mexican president says she told Donald Trump t...,french,0,0,"{'people': ['Donald Trump'], 'institutions': [...","[{'source': 'Mexico', 'target': 'US army', 're...",...,0,False,0.050000,False,other,0,None,False,0,True
3885,https://www.france24.com/fr/tag/pape-l%C3%A9on...,NaT,fr,Retrouvez toute l'actualité internationale et ...,,french,0,1,"{'people': ['Pape Léon XIV'], 'institutions': ...",[],...,0,False,0.000000,False,other,0,None,False,0,False
4040,https://www.spiegel.de/politik/deutschland/uni...,NaT,de,Union und SPD haben sich auf einen Koalitionsv...,Analyse: Der Koalitionsvertrag und die Klimapo...,german,1,0,"{'people': [], 'institutions': [], 'political_...",[],...,0,True,0.004149,False,other,0,None,False,0,False
4376,https://www.tagesschau.de/thema/livestream,NaT,de,Livestream - Nachrichten und Information: An 3...,Live-Shopping Zu Hause - und doch im Geschäft\...,german,0,0,"{'people': ['Karen Münster'], 'institutions': ...",[],...,0,False,0.000000,False,other,0,None,False,0,True
4766,https://www.faz.net/pro/weltwirtschaft/weltwis...,NaT,de,Über die Verflechtungen mit Deutschland treffe...,FAZ+ Osteuropa : Auch Trumps Freunde leiden un...,german,0,0,"{'people': ['Andreas Mihm', 'Trump'], 'institu...","[{'source': 'Trump', 'target': 'Osteuropa', 'r...",...,0,True,0.019231,False,other,0,None,False,0,False
4787,https://www.faz.net/podcasts/f-a-z-podcast-fue...,NaT,de,Ein Angelteich als Zentrale von Mafia-Drogenku...,"Wir sprechen darüber, wie weit verzweigt die k...",german,0,0,"{'people': ['Daniel Brombacher'], 'institution...",[],...,0,False,0.000000,False,other,0,None,False,0,False
4800,https://www.faz.net/aktuell/reise/maut-in-euro...,NaT,de,Einiges Europa? Von wegen. Die Gebühren für di...,"Ein Pickerl für Österreich, Maut für den Karaw...",german,0,0,"{'people': [], 'institutions': ['EU'], 'politi...",[],...,0,True,0.000000,False,other,0,None,False,0,False
4804,https://www.faz.net/aktuell/fotografie/wildpfe...,NaT,de,In der nordwestspanischen Region Galicien spie...,Wildpferde gegen das Feuer : Galiciens letzte ...,german,0,0,"{'people': [], 'institutions': ['Reuters'], 'p...",[],...,0,True,0.000000,False,other,0,None,False,0,False


In [237]:
df_featurized.columns

Index(['url', 'datetime', 'language', 'description', 'article_text', 'source',
       'is_relevant', 'used_description_as_text', 'extracted_entities',
       'entity_relationships', 'has_entities', 'has_relationships',
       'has_people', 'has_organizations', 'news_org', 'url_slug_title', 'date',
       'day_of_week', 'month', 'is_weekend', 'days_until_us_election',
       'days_until_french_election', 'days_until_german_election',
       'word_count', 'sentence_count', 'avg_sentence_length', 'noun_count',
       'verb_count', 'contains_question', 'quote_count', 'contains_number',
       'capital_ratio', 'mentions_social_media', 'article_type',
       'election_keyword_count', 'triggered_keyword',
       'mentions_election_keywords', 'election_certainty', 'no_title'],
      dtype='object')

# llm features 

In [252]:
import ast

# Total entities
df_featurized["L_n_total_entities"] = df_featurized["extracted_entities"].apply(
    lambda x: sum(len(v) for v in ast.literal_eval(x).values()) if pd.notna(x) else 0
)

# People, Locations, Political Actors, Institutions
df_featurized["L_n_people"] = df_featurized["extracted_entities"].apply(
    lambda x: len(ast.literal_eval(x).get("people", [])) if pd.notna(x) else 0
)
df_featurized["L_n_locations"] = df_featurized["extracted_entities"].apply(
    lambda x: len(ast.literal_eval(x).get("locations", [])) if pd.notna(x) else 0
)
df_featurized["L_n_institutions"] = df_featurized["extracted_entities"].apply(
    lambda x: len(ast.literal_eval(x).get("institutions", [])) if pd.notna(x) else 0
)


# Number of relationships
df_featurized["L_n_relationships"] = df_featurized["entity_relationships"].apply(
    lambda x: len(ast.literal_eval(x)) if pd.notna(x) else 0
)

In [253]:
df_featurized.groupby("is_relevant")[["L_n_total_entities","L_n_people", "L_n_locations", "L_n_institutions",  "L_n_relationships"]].describe().T

is_relevant                         0            1
L_n_total_entities count  4916.000000  1071.000000
                   mean     11.588283    20.641457
                   std      14.255071    26.076170
                   min       0.000000     0.000000
                   25%       3.000000     7.000000
                   50%       8.000000    15.000000
                   75%      16.000000    25.000000
                   max     255.000000   346.000000
L_n_people         count  4916.000000  1071.000000
                   mean      3.098861     6.844071
                   std       6.212381    12.242906
                   min       0.000000     0.000000
                   25%       0.000000     2.000000
                   50%       2.000000     4.000000
                   75%       4.000000     8.000000
                   max     142.000000   195.000000
L_n_locations      count  4916.000000  1071.000000
                   mean      4.193247     5.550887
                   std       5.482506     8.272179
                   min       0.000000     0.000000
                   25%       1.000000     1.000000
                   50%       2.000000     3.000000
                   75%       6.000000     7.000000
                   max     125.000000   115.000000
L_n_institutions   count  4916.000000  1071.000000
                   mean      3.646257     5.366013
                   std       4.628011     7.202578
                   min       0.000000     0.000000
                   25%       1.000000     2.000000
                   50%       2.000000     4.000000
                   75%       5.000000     7.000000
                   max      69.000000   158.000000
L_n_relationships  count  4916.000000  1071.000000
                   mean      2.978234     5.500467
                   std       4.362717     5.702742
                   min       0.000000     0.000000
                   25%       0.000000     1.000000
                   50%       1.000000     4.000000
                   75%       4.000000     8.000000
                   max      60.000000    72.000000

In [260]:
df_featurized["L_named_entity_density"] = df_featurized.apply(
    lambda row: (
        row["L_n_total_entities"] / len(str(
            row["article_text"] if pd.notna(row["article_text"]) and not row.get("used_description_as_text", 0)
            else row.get("description", "")
        ).split())
        if len(str(row["article_text"] if pd.notna(row["article_text"]) else row.get("description", "")).split()) > 0
        else 0
    ),
    axis=1
)

In [263]:
df_featurized.groupby("is_relevant")["L_named_entity_density"].describe()

,count,mean,std,min,25%,50%,75%,max
is_relevant,,,,,,,,
0,4916.0,0.031915,0.035713,0.0,0.000000,0.027027,0.043773,0.558824
1,1071.0,0.035156,0.027753,0.0,0.019538,0.028944,0.043421,0.198473


In [264]:
df_featurized["L_avg_sentiment_polarity"] = df_featurized["entity_relationships"].apply(
    lambda x: (
        np.mean([
            {"friendly": 1, "neutral": 0, "hostile": -1}.get(r.get("sentiment", "neutral").lower(), 0)
            for r in ast.literal_eval(x)
        ]) if pd.notna(x) and isinstance(ast.literal_eval(x), list) and len(ast.literal_eval(x)) > 0
        else 0
    )
)

In [265]:
df_featurized["L_avg_sentiment_polarity"].describe()

count    5987.000000
mean       -0.190431
std         0.395155
min        -1.000000
25%        -0.400000
50%         0.000000
75%         0.000000
max         1.000000
Name: L_avg_sentiment_polarity, dtype: float64

# encriching data set with publisher metadata 

# saving to csv

In [270]:
df_featurized.to_csv("data.csv")

In [269]:
df_featurized.news_org.value_counts()

news_org
www.aljazeera.com         663
www.politico.eu           641
www.euronews.com          590
www.france24.com          482
www.bbc.com               420
www.reuters.com           383
www.nbcnews.com           254
www.npr.org               218
www.cnn.com               209
www.liberation.fr         170
www.nytimes.com           170
www.zeit.de               170
www.lefigaro.fr           170
www.sueddeutsche.de       170
www.tagesschau.de         169
www.bild.de               169
www.rfi.fr                169
www.bfmtv.com             169
www.spiegel.de            169
www.lemonde.fr            168
www.faz.net               161
www.washingtonpost.com     66
www.bloomberg.com          18
apnews.com                 12
observers.france24.com      4
france24.com                2
www.theguardian.com         1
Name: count, dtype: int64

In [272]:
df_featurized.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5987 entries, 0 to 5986
Data columns (total 47 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   url                         5987 non-null   object        
 1   datetime                    5987 non-null   datetime64[ns]
 2   language                    5987 non-null   object        
 3   description                 5987 non-null   object        
 4   article_text                5983 non-null   object        
 5   source                      5987 non-null   object        
 6   is_relevant                 5987 non-null   Int64         
 7   used_description_as_text    5987 non-null   int64         
 8   extracted_entities          5987 non-null   object        
 9   entity_relationships        5987 non-null   object        
 10  has_entities                5987 non-null   int64         
 11  has_relationships           5987 non-null   int64       